# spokenform — Written Text to Spoken Form

`spokenform` converts written text in one explicitly selected language into
reviewable text intended for speech systems.

This notebook demonstrates the public API and includes an interactive playground.

**Binder users:** choose **Run → Run All Cells**, then scroll to the playground.
Changes made in a Binder session are temporary.

`spokenform` is a written-text normalization library, not a TTS engine: it does
not generate audio or phonemes. The caller explicitly selects the language; the
library does not detect input languages or automatically segment mixed-language text.

In [ ]:
from pathlib import Path

import spokenform
from spokenform import ProtectedSpan, prepare

print("spokenform version:", spokenform.__version__)
print("package path:", Path(spokenform.__file__).resolve())

## The smallest useful example

`prepare()` returns a structured `PreparedText` result. It contains the final
spoken text as well as stage and source-coordinate diagnostics.

In [ ]:
source = "Prof. Klein bringt am 14.05.2026 um 18:20 Uhr 2 kg mit."
prepared = prepare(source, language="de")

print(prepared.spoken_text)
print(prepared.render_changes())

## Multilingual examples

Each call processes one explicitly selected language or locale. The examples
use small expressions covered by the repository's behavior tests.

In [ ]:
examples = [
    ("en", "2+2=4"),
    ("de", "150€"),
    ("es_MX", "12-10-2023"),
    ("fr", "06 12 34 56 78"),
    ("it", "$25.50"),
    ("en", "2 g/cm³"),
]

print(f"{'language':<7} | {'source':<20} | spoken")
print("-" * 78)
for language, source in examples:
    result = prepare(source, language=language)
    print(f"{language:<7} | {source:<20} | {result.spoken_text}")

## Semantic and structured normalization

Structured values are recognized before broad number or symbol handling, so
the result reflects meaning and context rather than blindly spelling every
character. Quantities, currency, dates, math, music, biology, and identifiers
are all part of the public normalization surface.

In [ ]:
structured_examples = [
    ("en", "2 g/cm³"),
    ("de", "2,75%"),
    ("en", "6 L/100km"),
    ("en", "$1.5 million"),
    ("de", "150€"),
    ("en", "2025/03/15"),
    ("es_MX", "12-10-2023"),
    ("en", "2+2=4"),
    ("en", "chord C# and key Bb"),
    ("en", "E. coli and H2O"),
    ("en", "serial number: AB-123"),
    ("en", "Model E46"),
    ("en", "E46"),
]

for language, source in structured_examples:
    result = prepare(source, language=language)
    print(f"{language:>5} | {source:<30} → {result.spoken_text}")

## Generic acronym case policy

`generic_acronym_case` controls generic grapheme-spaced uppercase acronyms.
It is not a global uppercase/lowercase transformation: lexical acronyms,
known initialisms, identifiers, and special classes retain their own rules.

In [ ]:
source = "The stock symbol is AAPL."
upper = prepare(source, language="en", generic_acronym_case="upper")
lower = prepare(source, language="en", generic_acronym_case="lower")

print("upper:", upper.spoken_text)
print("lower:", lower.spoken_text)
print("ABC / upper:", prepare("ABC", language="en", generic_acronym_case="upper").spoken_text)
print("ABC / lower:", prepare("ABC", language="en", generic_acronym_case="lower").spoken_text)

## Residual symbol policy

Symbol filtering happens after semantic recognition.

- `symbol_mode="none"` preserves the backward-compatible behavior.
- `symbol_mode="remove"` removes remaining Unicode punctuation and symbols.
- `symbol_mode="keep"` removes residual symbols except exact codepoints in `keep_symbols`.

In [ ]:
source = "ABC: 12, (test)!"
default = prepare(source, language="en", symbol_mode="none")
remove_all = prepare(source, language="en", symbol_mode="remove")
keep_selected = prepare(
    source,
    language="en",
    symbol_mode="keep",
    keep_symbols=":;,()-,.",
)

print("none  :", default.spoken_text)
print("remove:", remove_all.spoken_text)
print("keep  :", keep_selected.spoken_text)

## Protected spans

Protection uses source coordinates. Caller-selected text is not rewritten by
semantic stages, which is safer than hiding content behind application-level
placeholders.

In [ ]:
source = "Keep Dr. unchanged, but speak 12."
start = source.index("Dr.")
protected = ProtectedSpan(start, start + len("Dr."))
result = prepare(source, language="en", protected_spans=[protected])

print(result.spoken_text)
print("protected spans:", result.protected_spans)
print("warnings:", result.warnings)

## Opt-in literal normalization

URLs, e-mail addresses, and version-like literals are conservatively
protected by default. Set `normalize_literals=True` when high-confidence
literal rendering is wanted. Caller-protected spans remain absolute.

In [ ]:
source = "Visit https://example.org/v2."
safe = prepare(source, language="en", normalize_literals=False)
promoted = prepare(source, language="en", normalize_literals=True)

print("default:", safe.spoken_text)
print("opt in :", promoted.spoken_text)

## Provenance and stage inspection

A `PreparedText` result keeps the intermediate stages and stable source
replacements. Inspecting only changed stages keeps diagnostics useful without
dumping every internal field.

In [ ]:
source = "Prof. Klein has 2 kg."
result = prepare(source, language="de")

print(result.render_changes())
print("\nChanged stages:")
for stage in result.stages:
    if stage.changed:
        print(stage.name)
        print("  before:", stage.before)
        print("  after :", stage.after)

print("\nSource replacements:")
for replacement in result.source_replacements:
    print(replacement)

## Source/output offset mapping

The structured result can map a source span into the final output and back.
This is useful when an application needs to preserve annotations or explain a
normalization decision to a user.

In [ ]:
source = "Prof. Klein has 2 kg."
result = prepare(source, language="de")

start = source.index("Prof.")
end = start + len("Prof.")
spoken_start, spoken_end = result.map_source_span(start, end)

print("source span:", source[start:end])
print("spoken span:", result.spoken_text[spoken_start:spoken_end])
print("reverse mapping:", result.map_output_span(spoken_start, spoken_end))

## Interactive playground

Edit the text and options below, then press **Normalize**. The callback uses
the same public `prepare()` function as the guided examples. It does not run
on every keystroke, and ordinary input/configuration errors are shown in the
output area.

In [ ]:
import json

import ipywidgets as widgets
from IPython.display import Markdown, display

EXAMPLES = {
    "English — math and identifiers": (
        "en",
        "Calculate 2+2=4. The serial number is AB-123 and the stock symbol is AAPL.",
    ),
    "German — dates, quantities, currency": (
        "de",
        "Prof. Klein bringt am 14.05.2026 2 kg für 150€.",
    ),
    "Spanish — date": (
        "es_MX",
        "La fecha es 12-10-2023.",
    ),
    "Italian — currency": (
        "it",
        "Il prezzo è $25.50.",
    ),
    "English — specialist expressions": (
        "en",
        "H₂O, E. coli, chord C#, and 2 g/cm³.",
    ),
}

example = widgets.Dropdown(
    options=list(EXAMPLES),
    value="English — math and identifiers",
    description="Example:",
)
language = widgets.Combobox(
    options=["en", "en_US", "en_GB", "de", "es", "es_MX", "fr", "it", "pt", "cs"],
    value="en",
    ensure_option=False,
    description="Language:",
)
text = widgets.Textarea(
    value=EXAMPLES[example.value][1],
    description="Text:",
    layout=widgets.Layout(width="100%", height="140px"),
)

expand_abbreviations = widgets.Checkbox(value=True, description="Expand abbreviations")
expand_structured = widgets.Checkbox(value=True, description="Expand structured values")
expand_numbers = widgets.Checkbox(value=True, description="Expand numbers")
normalize_literals = widgets.Checkbox(value=False, description="Normalize literals")
context = widgets.Checkbox(value=True, description="Use context")
symbol_mode = widgets.Dropdown(
    options=["none", "remove", "keep"],
    value="none",
    description="Symbol mode:",
)
keep_symbols = widgets.Text(
    value=":;,()-,.",
    description="Keep symbols:",
    disabled=True,
)
acronym_case = widgets.Dropdown(
    options=["upper", "lower"],
    value="upper",
    description="Acronym case:",
)
run = widgets.Button(description="Normalize", button_style="primary")
output = widgets.Output()

def select_example(change):
    if change["name"] != "value":
        return
    selected_language, selected_text = EXAMPLES[change["new"]]
    language.value = selected_language
    text.value = selected_text

def select_symbol_mode(change):
    if change["name"] == "value":
        keep_symbols.disabled = change["new"] != "keep"

def render_result(result):
    display(Markdown("### Spoken text"))
    print(result.spoken_text)

    if result.warnings:
        display(Markdown("### Warnings"))
        for warning in result.warnings:
            print("-", warning)

    display(Markdown("### Changed stages"))
    changed = [stage for stage in result.stages if stage.changed]
    if not changed:
        print("(none)")
    for stage in changed:
        print(f"[{stage.name}]")
        print(" ", stage.before)
        print("  ->", stage.after)

    display(Markdown("### Source replacements"))
    replacements = result.to_adapter_dict()["source_replacements"]
    print(json.dumps(replacements, ensure_ascii=False, indent=2))

def normalize_clicked(_):
    output.clear_output(wait=True)
    kwargs = {
        "language": language.value.strip(),
        "expand_abbreviations": expand_abbreviations.value,
        "expand_structured": expand_structured.value,
        "expand_numbers": expand_numbers.value,
        "normalize_literals": normalize_literals.value,
        "context": context.value,
        "symbol_mode": symbol_mode.value,
        "generic_acronym_case": acronym_case.value,
    }
    if symbol_mode.value == "keep":
        symbols = keep_symbols.value
        if not symbols:
            with output:
                print("keep_symbols must not be empty when symbol_mode='keep'")
            return
        kwargs["keep_symbols"] = symbols
    try:
        result = prepare(text.value, **kwargs)
    except Exception as exc:
        with output:
            print(f"{type(exc).__name__}: {exc}")
        return
    with output:
        render_result(result)

example.observe(select_example, names="value")
symbol_mode.observe(select_symbol_mode, names="value")
run.on_click(normalize_clicked)

In [ ]:
display(
    widgets.VBox(
        [
            example,
            language,
            text,
            widgets.HBox([expand_abbreviations, expand_structured]),
            widgets.HBox([expand_numbers, normalize_literals, context]),
            widgets.HBox([symbol_mode, keep_symbols, acronym_case]),
            run,
            output,
        ]
    )
)

## Optional spaCy annotations

`spokenform` can use an application-owned spaCy pipeline or an already-installed
model for source-aligned POS annotations. The public Binder image intentionally
keeps this optional dependency out so launches stay small and predictable.

If an application already has a compatible model, the public API looks like this:

```python
import spacy
from spokenform import prepare

nlp = spacy.load("en_core_web_sm")
result = prepare("The board is 2 in. wide.", language="en", nlp=nlp)
```

`spokenform` does not install models automatically, and the basic notebook
workflow does not require spaCy.